In [1]:
!pip install pyspark -q

In [2]:
import os
import shutil

# Create Bronze and Silver directories
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Move the uploaded sales.csv to the Bronze layer
if os.path.exists("sales.csv"):
    shutil.copy("sales.csv", "data_lake/bronze/sales.csv")
    print("sales.csv successfully moved to Bronze layer.")
else:
    print("WARNING: Please upload sales.csv to the Colab files pane first.")

sales.csv successfully moved to Bronze layer.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark
spark = SparkSession.builder \
    .appName("DataLakeAssignment") \
    .getOrCreate()

# Read the unchanged Bronze dataset
bronze_path = "data_lake/bronze/sales.csv"

df_bronze = spark.read.csv(
    bronze_path,
    header=True,
    inferSchema=True
)

print("Bronze data loaded successfully!")

Bronze data loaded successfully!


In [4]:
print("Bronze Data Schema:")
df_bronze.printSchema()

count_before = df_bronze.count()

print("\nTotal Bronze Records:", count_before)

print("\nFirst 5 Records:")
df_bronze.show(5, truncate=False)

Bronze Data Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)


Total Bronze Records: 1000

First 5 Records:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+-----

In [5]:
# ==========================================
# DATA QUALITY CHECKS
# ==========================================

# 1. Check exact duplicate rows
duplicate_count = count_before - df_bronze.dropDuplicates().count()

print("1. Exact duplicate rows:", duplicate_count)


# 2. Check missing values in every column
print("\n2. Missing values by column:")

null_counts = df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze.columns
])

null_counts.show()


# 3. Check text values for possible inconsistencies
print("\n3. Distinct Category values:")
df_bronze.select("category").distinct().show(truncate=False)

print("\nDistinct Payment Method values:")
df_bronze.select("payment_method").distinct().show(truncate=False)

print("\nDistinct City values:")
df_bronze.select("city").distinct().show(truncate=False)


# 4. Check invalid numerical values
print("\n4. Invalid Quantity values:")
df_bronze.filter(
    F.col("quantity").cast("int") <= 0
).show(truncate=False)

print("\nInvalid Unit Price values:")
df_bronze.filter(
    F.col("unit_price").cast("double") <= 0
).show(truncate=False)

print("\nInvalid Discount Percentage values:")
df_bronze.filter(
    (F.col("discount_percent").cast("double") < 0) |
    (F.col("discount_percent").cast("double") > 100)
).show(truncate=False)


# 5. Check invalid dates
print("\n5. Invalid Order Dates:")

date_check = df_bronze.withColumn(
    "parsed_date",
    F.to_date(F.col("order_date"))
)

date_check.filter(
    F.col("order_date").isNotNull() &
    F.col("parsed_date").isNull()
).select(
    "order_id",
    "order_date"
).show(truncate=False)

1. Exact duplicate rows: 10

2. Missing values by column:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+


3. Distinct Category values:
+---------------+
|category       |
+---------------+
|SPORTS         |
|beauty         |
|Sports         |
|Grocery        |
|Electronics    |
|Clothing       |
|Books          |
|ELECTRONICS    |
|Home Ap

{"ts": "2026-10-01 09:03:35.327", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value '31/02/2026' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "to_date", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o222.showString.\n: org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '31/02/2026' of the type \"STRING\" cannot be cast to \"DATE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"to_date\" was called from\nj

DateTimeException: [CAST_INVALID_INPUT] The value '31/02/2026' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"to_date" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)


In [6]:
# Safely check order dates without crashing on malformed dates

spark.conf.set("spark.sql.ansi.enabled", "false")

date_check = df_bronze.withColumn(
    "parsed_date",
    F.coalesce(
        F.to_date(F.col("order_date"), "yyyy-MM-dd"),
        F.to_date(F.col("order_date"), "dd/MM/yyyy")
    )
)

print("Invalid / unparseable order dates:")

date_check.filter(
    F.col("order_date").isNotNull() &
    F.col("parsed_date").isNull()
).select(
    "order_id",
    "order_date"
).show(truncate=False)

invalid_date_count = date_check.filter(
    F.col("order_date").isNotNull() &
    F.col("parsed_date").isNull()
).count()

print("Total invalid dates:", invalid_date_count)

Invalid / unparseable order dates:
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100358  |2025/07/14|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
|100969  |09-15-2026|
+--------+----------+

Total invalid dates: 6


In [7]:
# ==========================================
# SILVER LAYER PREPROCESSING
# ==========================================

# Start from the original Bronze data
df_silver = df_bronze

# ------------------------------------------
# 1. Remove exact duplicate rows
# ------------------------------------------
df_silver = df_silver.dropDuplicates()

# ------------------------------------------
# 2. Handle missing values
# ------------------------------------------
# Fill missing text values with "unknown"
text_fill_values = {
    "customer_name": "unknown",
    "payment_method": "unknown",
    "city": "unknown",
    "state": "unknown"
}

df_silver = df_silver.fillna(text_fill_values)

# ------------------------------------------
# 3. Clean and standardize text columns
# ------------------------------------------
text_columns = [
    "customer_id",
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for c in text_columns:
    df_silver = df_silver.withColumn(
        c,
        F.lower(F.trim(F.col(c)))
    )

# ------------------------------------------
# 4. Validate numerical columns
# ------------------------------------------
# Keep only positive quantities and prices
df_silver = df_silver.filter(
    (F.col("quantity") > 0) &
    (F.col("unit_price") > 0)
)

# Keep discount percentage between 0 and 100
df_silver = df_silver.filter(
    (F.col("discount_percent") >= 0) &
    (F.col("discount_percent") <= 100)
)

# ------------------------------------------
# 5. Standardize and validate order dates
# ------------------------------------------
# Support the different date formats present in the dataset
df_silver = df_silver.withColumn(
    "parsed_order_date",
    F.coalesce(
        F.to_date(F.col("order_date"), "yyyy-MM-dd"),
        F.to_date(F.col("order_date"), "yyyy/MM/dd"),
        F.to_date(F.col("order_date"), "dd/MM/yyyy"),
        F.to_date(F.col("order_date"), "MM-dd-yyyy")
    )
)

# Remove rows where the date cannot be parsed
df_silver = df_silver.filter(
    F.col("parsed_order_date").isNotNull()
)

# Replace the original date with the standardized date
df_silver = df_silver.drop("order_date") \
                     .withColumnRenamed("parsed_order_date", "order_date")

# ------------------------------------------
# 6. Cast numerical columns to appropriate types
# ------------------------------------------
df_silver = df_silver.withColumn(
    "quantity",
    F.col("quantity").cast("integer")
)

df_silver = df_silver.withColumn(
    "unit_price",
    F.col("unit_price").cast("double")
)

df_silver = df_silver.withColumn(
    "discount_percent",
    F.col("discount_percent").cast("double")
)

# ------------------------------------------
# Display cleaned data
# ------------------------------------------
print("Silver preprocessing completed successfully!")

print("\nCleaned Silver Schema:")
df_silver.printSchema()

print("\nFirst 5 cleaned records:")
df_silver.show(5, truncate=False)

Silver preprocessing completed successfully!

Cleaned Silver Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_status: string (nullable = true)
 |-- order_date: date (nullable = true)


First 5 cleaned records:
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+------------+----------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_status|order_date|
+

In [8]:
# ==========================================
# SILVER LAYER VERIFICATION
# ==========================================

count_after = df_silver.count()

print("Row count BEFORE preprocessing:", count_before)
print("Row count AFTER preprocessing:", count_after)
print("Rows removed:", count_before - count_after)

# Check that duplicates are gone
remaining_duplicates = count_after - df_silver.dropDuplicates().count()
print("\nRemaining duplicate rows:", remaining_duplicates)

# Check for remaining nulls
print("\nRemaining null values:")
df_silver.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
]).show()

# Check numerical validity
print("Invalid quantities remaining:",
      df_silver.filter(F.col("quantity") <= 0).count())

print("Invalid unit prices remaining:",
      df_silver.filter(F.col("unit_price") <= 0).count())

print("Invalid discounts remaining:",
      df_silver.filter(
          (F.col("discount_percent") < 0) |
          (F.col("discount_percent") > 100)
      ).count())

# Check invalid dates
print("Invalid dates remaining:",
      df_silver.filter(F.col("order_date").isNull()).count())

print("\nVerification completed.")

Row count BEFORE preprocessing: 1000
Row count AFTER preprocessing: 958
Rows removed: 42

Remaining duplicate rows: 0

Remaining null values:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_status|order_date|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|           0|         0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+

Invalid quantities remaining: 0
Invalid unit prices remaining: 0
Invalid discounts remaining: 0
Invalid dates remaining: 0

Verification completed.


In [9]:
# ==========================================
# SAVE CLEANED DATA TO SILVER LAYER
# ==========================================

silver_path = "data_lake/silver/preprocessed_sales"

# Save the cleaned DataFrame as CSV
df_silver.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_path)

print("Silver data saved successfully!")
print("Location:", silver_path)

Silver data saved successfully!
Location: data_lake/silver/preprocessed_sales


In [10]:
# ==========================================
# READ BACK SILVER DATA FOR FINAL VERIFICATION
# ==========================================

df_silver_read = spark.read.csv(
    silver_path,
    header=True,
    inferSchema=True
)

print("Silver records read back:", df_silver_read.count())

print("\nSilver Data Sample:")
df_silver_read.show(5, truncate=False)

Silver records read back: 958

Silver Data Sample:
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+------------+----------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_status|order_date|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+------------+----------+
|100331  |c0005      |sakshi joshi      |football              |sports   |2       |670.0     |30.0            |credit card     |bengaluru |karnataka  |cancelled   |2025-06-06|
|100333  |c0263      |siddharth kulkarni|programming book      |books    |4       |670.0     |0.0             |net banking     |pune      |maharashtra|delivered   |2025-05-08|
|100506  |c0228      |diya tripathi     |competitive exam guide|books

In [11]:
# Save Silver data as a single CSV file for submission

submission_path = "sales_silver.csv"

df_silver.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("silver_submission")

print("Silver submission file created successfully!")

Silver submission file created successfully!


In [12]:
import shutil
import os

# Find the generated Spark CSV file
files = os.listdir("silver_submission")
csv_file = [f for f in files if f.endswith(".csv")][0]

# Copy it to the required submission filename
shutil.copy(
    os.path.join("silver_submission", csv_file),
    "sales_silver.csv"
)

print("Created:", os.path.abspath("sales_silver.csv"))

Created: /content/sales_silver.csv


## Preprocessing Summary

The following preprocessing operations were performed on the Bronze dataset:

- Removed exact duplicate records.
- Handled missing values using appropriate preprocessing.
- Trimmed extra spaces and standardized text capitalization.
- Validated quantity and removed/handled invalid values.
- Validated unit price and removed/handled invalid values.
- Validated discount percentage to ensure values were between 0 and 100.
- Converted valid order dates to a consistent date format.
- Identified and handled invalid/unparseable dates.
- Saved the cleaned dataset in the Silver layer.
- Verified the saved Silver dataset by reading it back with Spark.